# Additional n=1 external validation prediction

to clean up and test

to check:

- only include ECoG and ipsilateral STN LFP
- exclude moments where was only Dyskinesia in body-side ipsilateral to ECoG (NOT CORRESPONDING WITH ECoG-hemisphere)

## Load packages and functions

In [ ]:
# Importing Python and external packages
import os
import importlib
import pandas as pd
import numpy as np
from itertools import compress

import matplotlib.pyplot as plt


In [ ]:
def get_project_path_in_notebook(
    subfolder: str = '',
):
    """
    Finds path of projectfolder from Notebook.
    Start running this once to correctly find
    other modules/functions
    """
    path = os.getcwd()

    while path[-20:] != 'dyskinesia_neurophys':

        path = os.path.dirname(path)
    
    return path

In [ ]:
# define local storage directories
projectpath = get_project_path_in_notebook()
codepath = os.path.join(projectpath, 'code')
figpath = os.path.join(projectpath, 'figures')
datapath = os.path.join(projectpath, 'data')
feat_path = os.path.join(projectpath, 'results', 'features')

In [ ]:
os.chdir(codepath)
# own utility functions
import utils.utils_fileManagement as utilsFiles
# own data exploration functions
import lfpecog_features.feats_read_proc_data as read_data
import lfpecog_preproc.preproc_import_scores_annotations as importClin
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_analysis.import_ephys_results as importResults
import lfpecog_analysis.stats_fts_lid_corrs as ftLidCorr
import lfpecog_analysis.load_SSD_features as load_ssdFts
import lfpecog_analysis.ft_processing_helpers as ftProc
import lfpecog_features.feats_helper_funcs as ftHelp
from lfpecog_features.get_ssd_data import get_subject_SSDs
import lfpecog_predict.prepare_predict_arrays as prep_pred_arrs

from lfpecog_plotting.plotHelpers import get_colors
import lfpecog_plotting.plotHelpers as pltHelp
import lfpecog_plotting.plot_FreqCorr as plotFtCorrs
import lfpecog_plotting.plot_SSD_feat_descriptives as plot_ssd_descr

import lfpecog_analysis.get_acc_task_derivs as accDerivs

## 1) Define data, feature settings, and create DataClasses

In [ ]:
# set variables
DATA_VERSION = 'v4.0'    # v4.0: new artef-rem, no reref; v3.0 multiple re-ref
FT_VERSION = 'v8'  # v4: broad-flanks, bursts; v3: broad-flanked SSD
INCL_PSD_FTS=['mean_psd', 'variation']
IGNORE_PTS = ['011', '104', '106']

CDRS_RATER = 'Patricia'
ANALYSIS_SIDE = 'BILAT'
INCL_CORE_CDRS = True
CATEG_CDRS = False
MILD_CDRS = 4
SEV_CDRS = 8

INCL_ECOG = False
INCL_ACC = True

# path were classes are saved and loaded from
extVal_path = os.path.join(utilsFiles.get_project_path('data'),
                           'ext_val_prediction')

for debugging single sub feat classes

In [ ]:
importlib.reload(load_ssdFts)


# get all available subs with features
SUBS = utilsFiles.get_avail_ssd_subs(DATA_VERSION=DATA_VERSION,
                                     FT_VERSION=FT_VERSION,
                                     IGNORE_PTS=IGNORE_PTS)
print(f'SUBS: n={len(SUBS)} ({SUBS})')

# use as single ft example to debug/develop
sub_fts = load_ssdFts.ssdFeatures(
    sub_list=['023', '024'],
    settings_json=f'ftExtr_spectral_{FT_VERSION}.json',
)

In [ ]:
sub_fts.sub024

Prepare Class with FEATS and CDRS-LABELS

In [ ]:
# LOAD FEATURE via FeatureClass containing all features
importlib.reload(utilsFiles)
importlib.reload(accDerivs)
importlib.reload(ftProc)
importlib.reload(importClin)
importlib.reload(load_ssdFts)
importlib.reload(ftLidCorr)


FeatLid = ftProc.FeatLidClass(
    FT_VERSION=FT_VERSION,
    CDRS_RATER=CDRS_RATER,
    INCL_ECOG=INCL_ECOG,
    INCL_ACC_RMS=INCL_ACC,
    EXCL_IPSI_ECOG=True,  # removes unilat ipsilat LID for ECOG-strip (ACC shape designed for this)
    CATEGORICAL_CDRS=CATEG_CDRS,
    CORR_TARGET='CDRS',
    cutMild=MILD_CDRS, cutSevere=SEV_CDRS,
    TO_CALC_CORR=False,
    verbose=True,
)

# print(f'features included: {FEATS[sub].keys()}') 

Save Classes

In [ ]:
# SAVE FeatLabelClass as pickle

className = f'featLabels_n{len(FeatLid.FEATS.keys())}_ft{FT_VERSION}'
if FeatLid.CORR_TARGET == 'LID': className += '_Lid'
elif FeatLid.CATEGORICAL_CDRS == True: className += '_CatCdrs'
else: className += '_Cdrs'

if FeatLid.INCL_ECOG: className += '_Ecog'
else: className += '_StnOnly'

if INCL_ACC: className += '_ACC'

utilsFiles.save_class_pickle(class_to_save=FeatLid,
                             path=extVal_path,
                             filename=className)

Import classes

In [ ]:
# LOAD existing classes with features and labels

if INCL_ECOG:
    fname_ext = '_Ecog'
    n_subs = 14
else:
    fname_ext = '_StnOnly'
    n_subs = 22

if INCL_ACC: fname_ext += '_ACC'

predData = utilsFiles.load_class_pickle(
    os.path.join(extVal_path,
                 f'featLabels_n{n_subs}_ft{FT_VERSION}_Cdrs{fname_ext}.P'),
    convert_float_np64=True
)


## 2) Prepare prediction arrays

Prepare X (features) and y (labels, LID) arrays for prediction analyses.

- 1) use all epochs, regardless of movement presence
- 2) make prediction movement aware, A) one classifier trains on and tests all epochs without (many) movements; B) one classifier trains on and tests epochs with more movement (movement-zscore-threshold is defined based on lineplot PPV/NPV)



Split training and external validation-test data


In [ ]:
dataDicts = {"train": {'FEATS': {}, 'LABELS': {}, 'ACC': {}},
             "extVal": {'FEATS': {}, 'LABELS': {}, 'ACC': {}}}

for sub in predData.FEATS.keys():
    if sub != '024':
        dataDicts['train']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['train']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['train']['ACC'][sub] = predData.ACC_RMS[sub]

    elif sub == '024':
        dataDicts['extVal']['FEATS'][sub] = predData.FEATS[sub]
        dataDicts['extVal']['LABELS'][sub] = predData.FT_LABELS[sub]
        dataDicts['extVal']['ACC'][sub] = predData.ACC_RMS[sub]

print('n-train', len(dataDicts['train']['FEATS'].keys()),
      '; n-validate', len(dataDicts['extVal']['FEATS'].keys()))



In [ ]:

def merge_pred_dicts_to_grouparrays(dataDict, accDict=False,):

    list_returns = prep_pred_arrs.get_group_arrays_for_prediction(
        feat_dict=dataDict['FEATS'],
        label_dict=dataDict['LABELS'],
        CDRS_CODING='binary',  # categorical
        acc_dict=accDict,
    )
    if len(list_returns) == 6:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names) = list_returns
        acc_total = False  # set false bool bcs absent
    elif len(list_returns) == 7:
        (X_total, y_total_binary, y_total_scale,
         sub_ids_total, ft_times_total, ft_names,
         acc_total) = list_returns

    # Merge subject-arrays to one group array for prediction
    list_returns = prep_pred_arrs.merge_group_arrays(
        X_total=X_total,
        y_total_binary=y_total_binary,
        y_total_scale=y_total_scale,
        sub_ids_total=sub_ids_total,
        ft_times_total=ft_times_total,
        ext_acc_arr=acc_total
    )
    if len(list_returns) == 5:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all) = list_returns
    elif len(list_returns) == 6:
        (X_all, y_all_binary, y_all_scale,
         sub_ids, ft_times_all, acc_total) = list_returns

    print(f'Subjects included ({len(np.unique(sub_ids))}): {np.unique(sub_ids)}')


    if accDict == False:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names

    else:
        return X_all, y_all_binary, sub_ids, ft_times_all, ft_names, acc_total




In [ ]:
# Create arrays per subject based on features and labels

importlib.reload(prep_pred_arrs)

(X_all, y_all_binary,
 sub_ids, ft_times_all,
 ft_names, acc_all) = {}, {}, {}, {}, {}, {}

for label in dataDicts.keys():

    (
        X_all[label],
        y_all_binary[label],
        sub_ids[label],
        ft_times_all[label],
        ft_names[label],
        acc_all[label]
    ) = merge_pred_dicts_to_grouparrays(
        dataDicts[label], accDict=dataDicts[label]['ACC']
    )



In [ ]:
for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

In [ ]:
def convert_features(X_arr, ft_names):
    """
    takes average over bilat lfp features
    takes average over single gamma bands into gammaBroad
    """
    X_df = pd.DataFrame(X_arr, columns=ft_names)

    # change unilat into mean bilat LFP powers per band
    for band in ['theta', 'alpha', 'lo_beta', 'hi_beta',
                'gamma1', 'gamma2', 'gamma3', 'gammaPeak']:
        for ft in ['mean_psd', 'variation']:
            # add columns with data
            X_df[f'lfp_mean_{band}_{ft}'] = np.mean(
                [X_df[f'lfp_left_{band}_{ft}'],
                X_df[f'lfp_right_{band}_{ft}']], axis=0
            )
            # drop unilat lfp columns
            for s in ['left', 'right']: X_df = X_df.drop(labels=[f'lfp_{s}_{band}_{ft}'], axis=1)
        
        # drop imagniary coh columns
        X_df = X_df.drop(labels=[f'imag_coh_STN_STN_{band}'], axis=1)

    # take average over broad gamma
    # select all columns containing single gamma bands
    gamma_cols = [f for f in X_df.keys() if 'gamma1' in f]

    for col in gamma_cols:
        # add mean gammaBroad and drop single gamma cols
        X_df[col.replace('gamma1', 'gammaBroad')] = np.mean(
                [X_df[col],
                X_df[col.replace('gamma1', 'gamma2')],
                X_df[col.replace('gamma1', 'gamma3')]], axis=0
            )
        # drop unilat lfp columns
        X_df = X_df.drop(labels=[col, col.replace('gamma1', 'gamma2'),
                                 col.replace('gamma1', 'gamma3')],
                         axis=1,)
    
    # rename ecog_right
    ftnames = [k.replace('ecog_right', 'ecog') for k in X_df.keys()]



    return X_df.values, ftnames

In [ ]:
# merge bilat STN features, merge broad gamma features
for label in X_all.keys():
    X_all[label], ft_names[label] = convert_features(X_all[label], ft_names[label])

for label in dataDicts.keys():
    
    print(label, X_all[label].shape, y_all_binary[label].shape)

print(ft_names['extVal'])

Explore movement splitting based on clustering

- Cluster has issues to differentiate small movements and rest, for now pragmatic -0.5 as cut off

In [ ]:
# plt.hist(acc_all['extVal'], bins=np.arange(-1, 4, 0.05))

# plt.show()

In [ ]:
# from sklearn.cluster import KMeans
# from sklearn.mixture import GaussianMixture  # better for skewed data

# # # KMeans clustering into 2 groups
# # kmeans = KMeans(n_clusters=2, random_state=0)
# # k_labels = kmeans.fit_predict(acc_all['extVal'].reshape(-1, 1))

# # # Optional: identify which label corresponds to rest vs movement
# # # Rest is the cluster with the lower mean RMS
# # cluster_means = [acc_all['extVal'][k_labels == i].mean() for i in range(2)]
# # rest_label = np.argmin(cluster_means)
# # movement_label = 1 - rest_label


# # Fit Gaussian Mixture Model with 2 components
# rms_values = v
# # nonlinear transformation on rms bcs of skewedness of data
# rms_values = np.sign(rms_values) * (np.abs(rms_values) ** 1.5)

# gmm = GaussianMixture(n_components=3, random_state=0,)
# gmm_labels = gmm.fit_predict(rms_values.reshape(-1, 1))

# # Identify rest vs movement based on component means
# gmm_means = gmm.means_.flatten()
# rest_label = np.argmin(gmm_means)
# movement_label = 1 - rest_label

In [ ]:
# lab='extVal'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.5, label='acc-rms',)

# # plt.scatter(ft_times_all[lab], k_labels + 3,
# #             s=10, alpha=.3, label='k-cluster',)
# plt.scatter(ft_times_all[lab], gmm_labels + 3,
#             s=10, alpha=.3, label='cluster labels',)


# plt.legend()
# plt.show()

In [ ]:
# lab='train'
# plt.scatter(ft_times_all[lab], y_all_binary[lab],
#             s=10, alpha=.5, label='LID',)
# plt.scatter(ft_times_all[lab], acc_all[lab],
#             s=10, alpha=.3, label='acc-rms',)

# plt.legend()
# plt.show()

## 3) Prediction

In [ ]:

from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
import gpboost as gpb

import joblib

# performance
from sklearn.metrics import (
    classification_report, accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay,
    auc, roc_curve, RocCurveDisplay
)

In [ ]:
import lfpecog_predict.predict_helpers as predHelpers
import lfpecog_plotting.plot_pred_standards as plotPred

In [ ]:
def plot_ext_validation(times, y_true, y_pred, y_probas,
                        ONLY_GAMMA, MOVE_AWARE, ONLY_STN, ONLY_ECOG,
                        acc_sig=[], noLID_col = 'green', LID_col = 'purple',
                        fs = 14,):
    if ONLY_STN: src = 'STN'
    elif ONLY_ECOG: src = 'ECOG'
    else: src = 'STNECOG'
    
    fname = f'extVal_Lda_{src}_allEpochs'
    if MOVE_AWARE: fname = fname.replace('allEpochs', 'moveAware')
    if ONLY_GAMMA: fname += '_onlyGamma'


    fig, axes = plt.subplots(1, 2, figsize=(12, 4),
                            gridspec_kw={'width_ratios': [2, 1]})

    # plot y-predictions, y-probabilities, y-true over time
    acc_score = accuracy_score(y_true=y_true, y_pred=y_pred)

    axes[0].scatter(times, y_probas[:, 0],
                    color=noLID_col, s=10, alpha=.5,
                    label='pred-proba-1',)
    axes[0].scatter(times, y_probas[:, 1],
                    color=LID_col, s=10, alpha=.5,
                    label='pred-proba-2',)
    
    # fill background for true LID state
    axes[0].fill_between(x=times, where=y_true==0,
                        y1=np.zeros(len(times)),
                        y2=np.ones(len(times)),
                        edgecolor=noLID_col, facecolor=noLID_col,
                        alpha=.25, label='true no LID',)
    axes[0].fill_between(x=times, where=y_true==1,
                        y1=np.zeros(len(times)),
                        y2=np.ones(len(times)),
                        edgecolor=LID_col, facecolor=LID_col,
                        alpha=.25, label='true LID',)
    
    # plot actual predictions
    axes[0].scatter(times, y_pred, color='orange',
                    label=f'Pred, accuracy: {np.round(acc_score, 2)}',)

    # plot acc signal
    if len(acc_sig) > 0:
        acc_ax = axes[0].twinx()
        acc_ax.plot(times, acc_sig, lw=2, color='gray', alpha=.5,)
        acc_ax.set_ylabel('Accelerometer vector (z-score)', fontsize=fs,)
        fname += '_acc'

    axes[0].set_xlabel('Time (min. vs L-DOPA intake)', size=fs,)
    axes[0].set_yticks([0, 1], size=fs,)
    axes[0].set_yticklabels(['No LID', 'LID'], size=fs,)

    axes[0].legend(ncol=3, bbox_to_anchor=[.0, 1.25],
                loc='upper left', fontsize=fs-2,
                frameon=False, )


    # plot AUROC
    fpr, tpr, _ = roc_curve(y_true, y_probas[:, 1])
    auc_score = round(auc(fpr, tpr), 2)

    axes[1].plot(fpr, tpr, c='darkgreen', lw=2,
            label=f'ROC, AUC: {auc_score}',
    )

    axes[1].plot([0, 1], [0, 1], lw=3,  c='orange', label='Chance level')

    axes[1].set_xlabel('False Positive Rate', fontsize=fs,)  #  weight='bold',
    axes[1].set_ylabel('True Positive Rate', fontsize=fs, )
    # axes[1].set_title('LID prediction - AUROC', fontsize=fs)

    axes[1].legend(frameon=False, fontsize=fs,
                loc='lower right',)

    for ax in axes:
        ax.tick_params(axis='both', labelsize=fs,)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()

    plt.savefig(os.path.join(figpath, 'prediction',
                             'ext_validation', fname),
                facecolor='w', dpi=300,)


    plt.show()


In [ ]:
def store_pred_results(
    pred_times, pred_y, true_y,
    MOVE_AWARE, ONLY_STN, ONLY_ECOG, ONLY_GAMMA,
):

    # store prediction results for statistics

    if MOVE_AWARE: cls_name = 'moveAware'
    else: cls_name = 'allEpochs'

    if ONLY_STN: cls_name += '_STN'
    elif ONLY_ECOG: cls_name += '_ECOG'
    else: cls_name += '_STNECOG'

    if ONLY_GAMMA: cls_name += '_onlyGamma'
    else: cls_name += '_allBands'



    pred_correct = pd.DataFrame(
        index=pred_times,
        data=pred_y == true_y,
        columns=[cls_name],
    )
    res_path = os.path.join(projectpath, 'results', 'ext_validation')
    fname = f'predCorrBool_{cls_name}.csv'
    pred_correct.to_csv(os.path.join(res_path, fname),
                        header=True, index=True,)
    
    print(f' saved {fname} to {res_path}')


In [ ]:
print(ft_names['extVal'])

#### Prediction with one classifier independent of movement-presence (movement unaware prediction)

In [ ]:
ONLY_GAMMA = False
ONLY_ECOG = False
ONLY_STN = True

assert not (ONLY_ECOG and ONLY_STN), "choose either ONLY STN or ECOG"

assert not (INCL_ECOG and ONLY_STN), "make sure you loaded ALL n=22 patients for ONLY STN"

# Model name and Path to store or load
if ONLY_GAMMA == False and ONLY_STN:
    model_name = 'extVal_lda01'
elif ONLY_GAMMA and ONLY_STN:
    model_name = 'extVal_lda02_onlyGamma'
elif ONLY_ECOG and ONLY_GAMMA == False:
    model_name = 'extVal_lda03_onlyEcog'
elif ONLY_ECOG and ONLY_GAMMA:
    model_name = 'extVal_lda04_onlyEcog_onlyGamma'
elif ONLY_ECOG==False and ONLY_STN==False and ONLY_GAMMA==False:
    model_name = 'extVal_lda05_StnEcog'
elif ONLY_ECOG==False and ONLY_STN==False:
    model_name = 'extVal_lda06_StnEcog_onlyGamma'
else:
    raise ValueError('incorrect variables defined')


print(f"modelname: {model_name}")
model_path = os.path.join(extVal_path, f'{model_name}.pkl')


# INCLUDE ALL
X_train = X_all['train']
y_train = y_all_binary['train']

X_test = X_all['extVal']
y_true_test = y_all_binary['extVal']

times_test = ft_times_all['extVal']


# SELECT FEATURES BASED ON MODEL
ftsel_idx = [True] * len(ft_names['train'])  # bool-array all True
if ONLY_GAMMA:
    ftsel_temp = ['gamma' in f for f in ft_names['train']]  # select gamma features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)
    
if ONLY_ECOG:
    ftsel_temp = [('ecog' in f.lower() and 'stn' not in f.lower())
                  for f in ft_names['train']]  # select ecog features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)

if ONLY_STN:
    ftsel_temp = ['ecog' not in f.lower() for f in ft_names['train']]  # exclude ecog features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)

sel_ftnames = list(compress(ft_names['train'], ftsel_idx))
print(f"select fts: {sel_ftnames}")
X_train = X_train[:, ftsel_idx]
X_test = X_test[:, ftsel_idx]


# Train LDA
lda = LDA()                 
lda.fit(X_train, y_train)

# # # Save model to disk
# joblib.dump(lda, model_path)


In [ ]:
# Load saved model
lda_loaded = joblib.load(model_path)
print(f'Model: {model_path} loaded')

# Predict binary LID presence
y_pred = lda_loaded.predict(X_test)
# predict probabilities
y_proba = lda_loaded.predict_proba(X_test)

In [ ]:
# store pred results for later statistics
store_pred_results(
    pred_times=times_test,
    pred_y=y_pred,
    true_y=y_true_test,
    MOVE_AWARE=False,
    ONLY_ECOG=ONLY_ECOG,
    ONLY_STN=ONLY_STN,
    ONLY_GAMMA=ONLY_GAMMA,
)

In [ ]:
plot_ext_validation(
    times=times_test,
    y_true=y_true_test,
    y_pred=y_pred,
    y_probas=y_proba,
    ONLY_STN=ONLY_STN,
    ONLY_ECOG=ONLY_ECOG,
    ONLY_GAMMA=ONLY_GAMMA,
    MOVE_AWARE=False,
    acc_sig=acc_all['extVal'],
)

#### Including move split for MOVEMENT AWARE PREDICTION

- make sure that for STN_ONLY, all train-patients are included (via EXCL ECOG) during loading of the pickle classes with features

In [ ]:
# set variables for movement aware prediction
ONLY_GAMMA = False
ONLY_ECOG = False
ONLY_STN = True

MOVE_AWARE = True  # always in this section

assert not (ONLY_ECOG and ONLY_STN), "choose either ONLY STN or ECOG"

# Model name and Path to store or load
if ONLY_GAMMA == False and ONLY_STN:
    restmodel_name = 'extVal_lda01rest'
    movemodel_name = 'extVal_lda01move'
elif ONLY_GAMMA and ONLY_STN:
    restmodel_name = 'extVal_lda02rest_onlyGamma'
    movemodel_name = 'extVal_lda02move_onlyGamma'
elif ONLY_ECOG and ONLY_GAMMA == False:
    restmodel_name = 'extVal_lda03rest_onlyEcog'
    movemodel_name = 'extVal_lda03move_onlyEcog'
elif ONLY_ECOG and ONLY_GAMMA:
    restmodel_name = 'extVal_lda04rest_onlyEcog_onlyGamma'
    movemodel_name = 'extVal_lda04move_onlyEcog_onlyGamma'
elif ONLY_ECOG==False and ONLY_STN==False and ONLY_GAMMA==False:
    restmodel_name = 'extVal_lda05rest_StnEcog'
    movemodel_name = 'extVal_lda05move_StnEcog'
elif ONLY_ECOG==False and ONLY_STN==False:
    restmodel_name = 'extVal_lda06rest_StnEcog_onlyGamma'
    movemodel_name = 'extVal_lda06move_StnEcog_onlyGamma'
else:
    raise ValueError('incorrect variables defined')

print(f"modelnames: {restmodel_name}, {movemodel_name}")
restmodel_path = os.path.join(extVal_path, f'{restmodel_name}.pkl')
movemodel_path = os.path.join(extVal_path, f'{movemodel_name}.pkl')

In [ ]:

### DEFINE MOVEMENT SPLIT
MOVE_CUT = -.5
MOVE_SPLIT_train = acc_all['train'] > MOVE_CUT  # True for rel movement
MOVE_SPLIT_test = acc_all['extVal'] > MOVE_CUT  # True for rel movement
print(f'total samples: {len(MOVE_SPLIT_test)}; '
      f'test-no-move, n = {sum(~MOVE_SPLIT_test)}; '
      f'test-move, n = {sum(MOVE_SPLIT_test)} '
      f'({round(sum(MOVE_SPLIT_test) / len(MOVE_SPLIT_test) * 100)} %)')


### SPLIT NO MOVEMENT DATA
X_train_A = X_all['train'][~MOVE_SPLIT_train]
y_train_A = y_all_binary['train'][~MOVE_SPLIT_train]

X_test_A = X_all['extVal'][~MOVE_SPLIT_test]
y_true_test_A = y_all_binary['extVal'][~MOVE_SPLIT_test]
times_test_A = ft_times_all['extVal'][~MOVE_SPLIT_test]

### SPLIT MOVEMENT DATA
X_train_B = X_all['train'][MOVE_SPLIT_train]
y_train_B = y_all_binary['train'][MOVE_SPLIT_train]

X_test_B = X_all['extVal'][MOVE_SPLIT_test]
y_true_test_B = y_all_binary['extVal'][MOVE_SPLIT_test]
times_test_B = ft_times_all['extVal'][MOVE_SPLIT_test]


### SELECT FEATURES BASED ON MODEL
ftsel_idx = [True] * len(ft_names['train'])  # bool-array all True
if ONLY_GAMMA:
    ftsel_temp = ['gamma' in f for f in ft_names['train']]  # select gamma features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)
    
if ONLY_ECOG:
    ftsel_temp = [('ecog' in f.lower() and 'stn' not in f.lower())
                  for f in ft_names['train']]  # select ecog features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)

if ONLY_STN:
    ftsel_temp = ['ecog' not in f.lower() for f in ft_names['train']]  # exclude ecog features
    ftsel_idx = np.logical_and(ftsel_idx, ftsel_temp)

sel_ftnames = list(compress(ft_names['train'], ftsel_idx))
print(f"select fts: {sel_ftnames}")

X_train_A = X_train_A[:, ftsel_idx]  # select train-nomove features
X_test_A = X_test_A[:, ftsel_idx]  # select test-nomove features

X_train_B = X_train_B[:, ftsel_idx]  # select train-move features
X_test_B = X_test_B[:, ftsel_idx]  # select test-move features


### Train and Store LDA-classifiers

# train no-movement classifier
lda = LDA()
lda.fit(X_train_A, y_train_A)
# Save model to disk
joblib.dump(lda, restmodel_path)

# train movement classifier
lda = LDA()
lda.fit(X_train_B, y_train_B)
# Save model to disk
joblib.dump(lda, movemodel_path)



Load Models and Predict


In [ ]:
# Load saved model rest
restlda_loaded = joblib.load(restmodel_path)
print(f'NoMove-Model: {restmodel_name} loaded')

# Load saved model move
movelda_loaded = joblib.load(movemodel_path)
print(f'Move-Model: {movemodel_name} loaded')

In [ ]:
# Predict no-movement
y_pred_A = restlda_loaded.predict(X_test_A)  # get binary predictions
y_proba_A = restlda_loaded.predict_proba(X_test_A)  # get probabilities

# Predict movement
y_pred_B = movelda_loaded.predict(X_test_B)  # get binary predictions
y_proba_B = movelda_loaded.predict_proba(X_test_B)  # get probabilities


# merge and sort Prediction-results-arrays from rest and move models
y_times_AB = np.concatenate([times_test_A, times_test_B])
t_sort_idx = np.argsort(y_times_AB)   # sort on chronological-times
merged_t = y_times_AB[t_sort_idx]

merged_y_pred = np.concatenate([y_pred_A, y_pred_B])[t_sort_idx]
merged_y_proba = np.concatenate([y_proba_A, y_proba_B])[t_sort_idx]
merged_y_true_test = np.concatenate([y_true_test_A, y_true_test_B])[t_sort_idx]


In [ ]:
store_pred_results(
    pred_times=merged_t,
    pred_y=merged_y_pred,
    true_y=merged_y_true_test,
    MOVE_AWARE=MOVE_AWARE,
    ONLY_ECOG=ONLY_ECOG,
    ONLY_STN=ONLY_STN,
    ONLY_GAMMA=ONLY_GAMMA,
)

Plot Movement Aware Models

In [ ]:
plot_ext_validation(
    times=merged_t,
    y_true=merged_y_true_test,
    y_pred=merged_y_pred,
    y_probas=merged_y_proba,
    ONLY_STN=ONLY_STN,
    ONLY_ECOG=ONLY_ECOG,
    ONLY_GAMMA=ONLY_GAMMA,
    MOVE_AWARE=True,
    acc_sig=acc_all['extVal'],
)

#### Statistically compare different classifiers

In [ ]:
res_path = os.path.join(projectpath, 'results', 'ext_validation')

filelist = [f for f in os.listdir(res_path) if f.startswith('predCorrBool')]


for i_f, f in enumerate(filelist):
    # create df with first file
    if i_f == 0:
        res_df = pd.read_csv(os.path.join(res_path, f), index_col=0)
    else:
        temp_df = pd.read_csv(os.path.join(res_path, f), index_col=0)
        assert all(res_df.index == temp_df.index)
        newcol = temp_df.keys()[0]

        res_df[newcol] = temp_df[newcol]

In [ ]:
for k in res_df.keys():
    acc = sum(res_df[k]) / res_df.shape[0]
    print(f'for {k} accuracy is {round(acc, 2)}')

In [ ]:
xlabel_ticks = {'STN': 'STN_allBands',
                'STN-$\gamma$': 'STN_onlyGamma',
                'Cortex': 'ECOG_allBands',
                'Cortex-$\gamma$': 'ECOG_onlyGamma',
                'STN-\nCortex': 'STNECOG_allBands',
                'STN-\nCortex-$\gamma$': 'STNECOG_onlyGamma'}

acc_list = []
movAwe_list = []
allEp_list = []

# add first allEpochs, then moveAware
for lab, k in xlabel_ticks.items():
    
    key = f'allEpochs_{k}'
    acc = sum(res_df[key]) / res_df.shape[0]
    acc_list.append(acc)
    allEp_list.append(acc)
    # add move aware
    key = f'moveAware_{k}'
    acc = sum(res_df[key]) / res_df.shape[0]
    acc_list.append(acc)
    movAwe_list.append(acc)
    

Plot classifier-accuracy overview

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(8, 3))
fs = 14

colors = ['olive', 'darkorange']

# xticks = []
# for k in np.arange(len(xlabel_ticks)):
#     xticks.extend([k - .15, k + .15])
# ax.bar(x=xticks, height=acc_list, width=.3,)


xticks = np.arange(len(xlabel_ticks))
ax.bar(x=xticks, height=allEp_list, width=-.3, align='edge',
       color=colors[0], alpha=.6, label='movement naive',)
ax.bar(x=xticks, height=movAwe_list, width=.3, align='edge',
       color=colors[1], alpha=.6, label='movement aware',)

# add significances (acc to McNemar Test below)
sign_list = [False, True, True, True, False, True]
for i_sig, (y_star, sign) in enumerate(zip(movAwe_list, sign_list)):
       if sign: ax.scatter(i_sig+.15, y_star + .05, marker='*',
                           c='k', alpha=.8, s=50,)

ax.set_xticks(np.arange(len(xlabel_ticks)),)
ax.set_xticklabels(xlabel_ticks)

ax.set_yticks([0, 0.5, 1])
ax.set_yticklabels([0, 50, 100])
ax.set_ylabel('Classifier accuracy (%)', size=fs,)
for yline in [0.25, .5, .75]:
    ax.axhline(yline, xmin=0, xmax=1, lw=1,
               alpha=.3, c='gray', zorder=-1,)

ax.legend(ncol=2,fontsize=fs, frameon=False,
          bbox_to_anchor=[.95, 1.1], loc='upper right',)

ax.tick_params(axis='both', size=fs, labelsize=fs,)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()

plt.savefig(os.path.join(figpath, 'prediction',
                         'ext_validation', 'acc_bars.pdf'),
              facecolor='w', dpi=300,)
plt.close()

Panel Plot ext validation

In [ ]:
def plot_ext_validation_panel(
    times, y_true, y_pred, y_probas, acc_sig,
    noLID_col = 'green', LID_col = 'purple', fs = 14,
):
    
    fname = f'extVal_Panel_STN_allBands'


    fig, axes = plt.subplots(2, 1, figsize=(12, 4),
                             gridspec_kw={'height_ratios': [5, 1]},
                             sharex=True,)

    true_ax = axes[1]
    pred_ax = axes[0]
    # # plot y-predictions, y-probabilities, y-true over time
    # acc_score = accuracy_score(y_true=y_true, y_pred=y_pred)

    # ax.scatter(times, y_probas[:, 0],
    #             color=noLID_col, s=10, alpha=.3,
    #             label='pred-proba-1',)
    # ax.scatter(times, y_probas[:, 1],
    #             color=LID_col, s=10, alpha=.3,
    #             label='pred-proba-2',)

    pred_ax.bar(times, y_probas[:, 0], width=.09, 
           color=noLID_col, alpha=.3,
           label='None Dyskinetic',)
    pred_ax.bar(times, y_probas[:, 1], bottom=y_proba[:, 0],
            color=LID_col, alpha=.3, width=.09, 
            label='Dyskinetic',)
    
    # fill background for true LID state
    # ax.fill_between(x=times, where=y_true==0,
    #                 y1=np.zeros(len(times)),
    #                 y2=np.ones(len(times)),
    #                 edgecolor=noLID_col, facecolor=noLID_col,
    #                 alpha=.15, label='true no LID',)
    # ax.fill_between(x=times, where=y_true==1,
    #                 y1=np.zeros(len(times)),
    #                 y2=np.ones(len(times)),
    #                 edgecolor=LID_col, facecolor=LID_col,
    #                 alpha=.15, label='true LID',)
    
    # plot actual predictions
    # ax.scatter(
    #     times, y_pred, color='orange',
    #     # label=f'Pred, accuracy: {np.round(acc_score, 2)}',
    # )

    # plot acc signal
    acc_ax = pred_ax.twinx()
    
    # shift_acc = abs(np.min(acc_sig))  # shift to make all bars positive
    # acc_sig += shift_acc
    # acc_ax.bar(times, height=acc_sig, width=.5, color='gray', alpha=.3,)
    
    # prevent acc-connecting lines in empty minutes
    # uniq_minutes = np.unique(np.round(times))
    # min_range = np.arange(uniq_minutes[0], uniq_minutes[-1])
    # fill_mins = [m for m in min_range if m not in uniq_minutes]
    fill_mins = [t for t, d in zip(times, np.diff(times)) if d > .2]
    acc_times = times.copy()
    for m in fill_mins:
        # i_min = np.where(acc_times > m)[0][0]
        i_min = np.where(acc_times == m)[0][0] + 1
        acc_times = np.insert(acc_times, i_min, m)
        acc_sig = np.insert(acc_sig, i_min, np.nan)

    acc_ax.plot(acc_times, acc_sig, lw=3, color='gray', alpha=.5,
                label='Movement presence')
    pred_ax.plot([], [], lw=3, color='gray', alpha=.5,
                label='Movement presence')
    
    acc_ax.set_ylabel('Accelerometer vector (sd)', fontsize=fs,)


    axes[1].set_xlabel('Time (min. vs L-DOPA intake)', size=fs,)
    # ax.set_yticks([0, 1], size=fs,)
    # ax.set_yticklabels(['No LID', 'LID'], size=fs,)
    pred_ax.set_ylabel('Prediction\nprobability (a.u.)', size=fs,)


    for ax_temp in [pred_ax, acc_ax]:
        ax_temp.tick_params(axis='both', labelsize=fs,)
        ax_temp.spines['top'].set_visible(False)
        # ax_temp.spines['right'].set_visible(False)
    
    
    true_ax.fill_between(x=times, where=y_true==0,
                    y1=np.zeros(len(times)),
                    y2=np.ones(len(times)),
                    edgecolor=noLID_col, facecolor=noLID_col,
                    alpha=.3, label='None Dyskinetic')
    true_ax.fill_between(x=times, where=y_true==1,
                    y1=np.zeros(len(times)),
                    y2=np.ones(len(times)),
                    edgecolor=LID_col, facecolor=LID_col,
                    alpha=.3, label='Dyskinetic',)

    i_onset = np.where(y_true_test == 1)[0][0]
    t_onset = times[i_onset]
    for ax in axes:
        ax.axvline(x=t_onset, ymin=0, ymax=1,color=LID_col, lw=2, alpha=.8,
                    label='true LID onset')

    pred_ax.legend(
            #   ncol=3, loc='upper left', bbox_to_anchor=[.0, 1.25],
              ncol=4, loc='upper center', bbox_to_anchor=[.5, 1.2],
              fontsize=fs, frameon=False, )

    # fix labels, spines
    true_ax.tick_params(axis='both', labelsize=fs,)
    # true_ax.set_xticks([])
    # true_ax.set_xticklabels([])
    for spine in true_ax.spines.values(): spine.set_visible(False)

    # true_ax.axis('off')
    true_ax.set_yticks([.5], size=fs,)
    true_ax.set_yticklabels(['true state'], fontsize=fs,)
    # true_ax.set_ylabel('true state', size=fs, rotation=0,)
    

    plt.tight_layout()

    plt.savefig(os.path.join(figpath, 'prediction',
                             'ext_validation', fname),
                facecolor='w', dpi=300,)


    plt.show()


In [ ]:
plot_ext_validation_panel(
    times=times_test,
    y_true=y_true_test,
    y_pred=y_pred,
    y_probas=y_proba,
    acc_sig=acc_all['extVal'],
)


Test for significant difference between the correctness of models

In [ ]:
from itertools import product

In [ ]:
"""
use McNemar Test to compare the paired binary prediction outcomes
Fagerland et al. BMC Med Res Methodology 2013.
https://doi.org/10.1186/1471-2288-13-91
"""
from statsmodels.stats.contingency_tables import mcnemar

In [ ]:
alpha = .05
mult_comp = 6
alpha /= mult_comp
print(f'alpha corrected is {alpha}\n')

for src, bands in product(['STN', 'ECOG', 'STNECOG'],
                          ['allBands', 'onlyGamma']):
    
    y1 = res_df[f'allEpochs_{src}_{bands}']
    y2 = res_df[f'moveAware_{src}_{bands}']

    # Create 2x2 table for McNemar's:
    #           y2=1   y2=0
    # y1=1   |   a   |   b   |
    # y1=0   |   c   |   d   |
    a = np.sum((y1 == 1) & (y2 == 1))
    b = np.sum((y1 == 1) & (y2 == 0))
    c = np.sum((y1 == 0) & (y2 == 1))
    d = np.sum((y1 == 0) & (y2 == 0))

    table = [[a, b],
            [c, d]]

    result = mcnemar(table, exact=True)
    print(f"Movement-Aware vs -Naiv for {src} "
          f"with {bands}, p-value = {round(result.pvalue, 8)}"
          f"\tSignificant is {result.pvalue < alpha}")
    # print(result)
    print('\n')

Conf Matrix

In [ ]:
# importlib.reload(plotPred)


# # show metrics summary
# print(classification_report(y_true_all, y_pred_all))

# # show confusion matrix
# cm = confusion_matrix(y_true_all, y_pred_all)
# cm_figname = 'Group_LID_Pred_LDA_powCoh_confMatrix'
# # plotPred.plot_confMatrix(cm, fig_path=figpath, fig_name=cm_figname,
# #                          to_show=False, to_save=True)

# # show Receiver Operator Cruve
# fpr, tpr, _ = roc_curve(y_true_all, y_pred_conf_all,)
# auc_score = auc(fpr, tpr)
# acc_score = accuracy_score(y_true_all, y_pred_all)
# print(f'AUC: {round(auc_score, 3)}, Accuracy: {round(acc_score, 3)}')
# # roc_display = RocCurveDisplay(fpr=fpr, tpr=tpr).plot()
